# SPARQL-Relax Tutorial

This notebook demonstrates how to use the `sparql-relax` Python bindings to diagnose and repair broken SPARQL queries.

## What is SPARQL-Relax?
SPARQL-Relax helps find why a SPARQL query returns no results when it's expected to, and attempts to find a "connected" version of the query that does return results by searching for paths in the knowledge graph.

In [1]:
from sparql_relax import diagnose, diagnose_and_connect, query, Store
import os

## 1. Setup Data

First, we load our knowledge graph (TTL file). We'll use the `b59` building dataset.

In [2]:
data_path = "/home/lazlo/Desktop/semantics/sparql-relax/eval/buildings/b59.ttl"
with open(data_path, "r") as f:
    data = f.read()
print(f"Loaded data from {data_path}")

Loaded data from /home/lazlo/Desktop/semantics/sparql-relax/eval/buildings/b59.ttl


## 2. Running a Correct Query

Let's start with a query that we know works.

In [3]:
correct_query = """
PREFIX s223: <http://data.ashrae.org/standard223#>
PREFIX ex1: <http://data.ashrae.org/standard223/data/lbnl-example-2#>
PREFIX qudt: <http://qudt.org/schema/qudt/>
PREFIX ref: <https://brickschema.org/schema/Brick/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX quantitykind: <http://qudt.org/vocab/quantitykind/>

SELECT DISTINCT ?target ?label ?alc_endpoint ?mpc_endpoint WHERE {
{ ?target a s223:Zone; s223:hasProperty ?prop. }
UNION
{ ?target a s223:Zone; s223:hasDomainSpace ?space .
?space a ex1:RoomDomainSpace ;
s223:connected/s223:connected/s223:contains/s223:hasProperty ?prop . }

?prop a s223:Property;
qudt:hasQuantityKind|s223:hasEnumerationKind ?quantKind;
rdfs:label ?label;
s223:hasExternalReference ?ref .

?ref ref:endpoint ?alc_endpoint;
ref:mpc_db_name ?mpc_endpoint;
ref:writable "False" .
}
"""

results = query(data, correct_query)
print(f"Found {len(results.bindings)} results.")
for row in results.bindings[:5]:
    print(row)


Found 354 results.
{'target': Term(kind='uri', value='http://data.ashrae.org/standard223/data/lbnl-example-2#04500', datatype=None, language=None), 'label': Term(kind='literal', value='core_zone_remote_air_temp', datatype='http://www.w3.org/2001/XMLSchema#string', language=None), 'alc_endpoint': Term(kind='literal', value='#lbnl_59-bl-072/remote_zone_temp', datatype='http://www.w3.org/2001/XMLSchema#string', language=None), 'mpc_endpoint': Term(kind='literal', value='zone_072c_temp', datatype='http://www.w3.org/2001/XMLSchema#string', language=None)}
{'target': Term(kind='uri', value='http://data.ashrae.org/standard223/data/lbnl-example-2#04500', datatype=None, language=None), 'label': Term(kind='literal', value='zone_co2', datatype='http://www.w3.org/2001/XMLSchema#string', language=None), 'alc_endpoint': Term(kind='literal', value='#lbnl_59-bl-072/zone_co2', datatype='http://www.w3.org/2001/XMLSchema#string', language=None), 'mpc_endpoint': Term(kind='literal', value='zone_072_co2', 

## 3. Diagnosing a Broken Query

Now, let's take a query that is slightly broken (e.g., uses a wrong class). This query should return no results.

In [4]:
broken_query = """
PREFIX s223: <http://data.ashrae.org/standard223#>
PREFIX ex1: <http://data.ashrae.org/standard223/data/lbnl-example-2#>
PREFIX qudt: <http://qudt.org/schema/qudt/>
PREFIX ref: <https://brickschema.org/schema/Brick/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX quantitykind: <http://qudt.org/vocab/quantitykind/>

SELECT DISTINCT ?target ?label ?alc_endpoint ?mpc_endpoint WHERE {
{ ?target a s223:Zone; s223:hasProperty ?prop. }
UNION
{ ?target a s223:Zone; s223:hasDomainSpace ?space .
?space a ex1:RoomDomainSpace ;
s223:connected/s223:connected/s223:contains/s223:hasProperty ?prop . }

?prop a s223:Point; # ERROR ADDED HERE
qudt:hasQuantityKind|s223:hasEnumerationKind ?quantKind;
rdfs:label ?label;
s223:hasExternalReference ?ref .

?ref ref:endpoint ?alc_endpoint;
ref:mpc_db_name ?mpc_endpoint;
ref:writable "False" .
}
"""

results = query(data, broken_query)
print(f"Found {len(results.bindings)} results (expected 0).")

print("Diagnosing...")
diagnosis = diagnose(data, broken_query)
for culprit in diagnosis.culprits:
    print(f"Culprit found: {culprit}")

Found 0 results (expected 0).
Diagnosing...
Culprit found: Culprit(triples=['?prop <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://data.ashrae.org/standard223#Point>'], depth=1)


## 4. Connecting the Query

Now we use `diagnose_and_connect` to automatically find a fix for the broken query.

In [5]:
print("Connecting query...")
report = diagnose_and_connect(data, broken_query)

for result in report.results:
    if result.fixed:
        print(f"Fixed triple: {result.triple} -> {result.path_text}")
        print("Connected Query:\n", result.connected_query)
        
        # Verify the fixed query
        fixed_results = query(data, result.connected_query)
        print(f"Fixed query found {len(fixed_results.bindings)} results.\n")

Connecting query...


### Ignoring cartesian risk

Some broken combinations are skipped by default because checking them could force a costly cross-product evaluation. Pass `ignore_cartesian_risk=True` to force the check anyway.

In [6]:
incorrect_path_query = """
PREFIX s223: <http://data.ashrae.org/standard223#>
PREFIX ex1: <http://data.ashrae.org/standard223/data/lbnl-example-2#>
PREFIX qudt: <http://qudt.org/schema/qudt/>
PREFIX ref: <https://brickschema.org/schema/Brick/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX quantitykind: <http://qudt.org/vocab/quantitykind/>

SELECT DISTINCT ?target ?label ?alc_endpoint ?mpc_endpoint WHERE {
{ ?target a s223:Zone; s223:hasProperty ?prop. }
UNION
{ ?target a s223:Zone; s223:hasDomainSpace ?space .
?space a ex1:RoomDomainSpace ;
s223:connected/s223:connected/s223:contains/s223:hasProperty ?prop . }

?prop a s223:Property;
qudt:hasQuantityKind|s223:hasEnumerationKind ?quantKind;
rdfs:label ?label;
ref:hasExternalReference ?ref . # INCORRECT NAMESPACE on ref:hasExternalReference

?ref ref:endpoint ?alc_endpoint;
ref:mpc_db_name ?mpc_endpoint;
ref:writable "False" .
}
"""

results = query(data, incorrect_path_query)
print(f"Found {len(results.bindings)} results (expected 0).")

print("Connecting...")
diagnosis = diagnose_and_connect(data, incorrect_path_query, ignore_cartesian_risk=True,max_depth = 3)
print(diagnosis)

Found 0 results (expected 0).
Connecting...
ConnectReport(original_row_count=0, results=[ConnectedCulprit(found_at_depth=1, triples=[ConnectedTriple(triple='?prop <https://brickschema.org/schema/Brick/hasExternalReference> ?ref', path_text='((<http://data.ashrae.org/standard223#hasExternalReference> / <https://brickschema.org/schema/Brick/writable>) / ^(<https://brickschema.org/schema/Brick/writable>))')], connected_query='SELECT DISTINCT ?target ?label ?alc_endpoint ?mpc_endpoint WHERE { { ?target <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://data.ashrae.org/standard223#Zone> .?target <http://data.ashrae.org/standard223#hasProperty> ?prop . } UNION { ?target <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://data.ashrae.org/standard223#Zone> .?target <http://data.ashrae.org/standard223#hasDomainSpace> ?space .?space <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://data.ashrae.org/standard223/data/lbnl-example-2#RoomDomainSpace> .?space <http://data.ashrae.o

In [7]:
diagnosis.results[0].triples

[ConnectedTriple(triple='?prop <https://brickschema.org/schema/Brick/hasExternalReference> ?ref', path_text='((<http://data.ashrae.org/standard223#hasExternalReference> / <https://brickschema.org/schema/Brick/writable>) / ^(<https://brickschema.org/schema/Brick/writable>))')]

## 5. Efficient Usage with Store

If you are running multiple queries against the same data, use a `Store` to avoid reparsing the TTL file every time.

In [8]:
store = Store(data)
results = store.query(correct_query)
print(f"Store query found {len(results.bindings)} results.")

Store query found 354 results.
